# STEP00. 데이터 준비: SSIS 서비스 마스터와 2023 은둔 수요 테이블

## 분석 질문
분석(Q1~Q3)과 LLM 추천이 함께 쓰는 **서비스 1행 마스터**와 **은둔 청년 수요 테이블**을 재현 가능하게 만든다.
- Q1 필요한 지원에 맞는 복지서비스가 존재하는가 / Q2 정보가 분산되거나 불완전한가 / Q3 기존 검색은 무엇을 놓치거나 잘못 판단하는가 — 이 단계는 질문에 답하지 않고 입력만 만든다.
- 선행 단계: 없음(수집 결과 `data/raw/`에서 시작). 후속: `notebooks/01_eda.ipynb`.
- 설계 기준: Notion '구현 계획'(3차 회의)을 반영한 작업 지시서. 이 노트북 작성 중 Notion 페이지를 다시 열람하지는 않았다.

## 입력
| 구분 | 프로젝트 상대경로 | 버전·단위·용도 |
|---|---|---|
| SSIS 중앙 목록·상세 | `data/raw/ssis/central_list.csv`, `central_detail_full.csv` | 2026-10-30 수집, 서비스 461건 |
| SSIS 지자체 목록·상세 | `data/raw/ssis/local_list.csv`, `local_detail_full.csv` | 2026-10-30 수집, 서비스 4,827건 |
| 보조금24 | `data/raw/gov24/gov24_*.csv` | 2026-10-31 수집. Q2 비교 출처, 이번에는 로딩·품질 점검만 |
| 2023 고립·은둔 청년 실태조사 | `data/raw/isolated_youth_survey_2023/*.xlsx` | 1차조사·은둔·고립 시트, 가중치 없음 |
| 2024 청년삶 파일설계서 | `docs/survey_codebooks/youth_life_survey/` 아래 2024 파일설계서(xlsx) | 358(외출 빈도)·360(은둔 생활 이유) 문항 근거 |

## 출력
| 유형 | 경로 | 내용 | 상태 |
|---|---|---|---|
| 데이터 | `data/processed/services_master.csv` | 서비스 1행, (A)~(G) | 생성 (Git 제외) |
| 데이터 | `data/processed/demand_2023_hikikomori.csv` | 은둔 4,448명 Q21 | 생성 (Git 제외) |
| 데이터 | `data/processed/demand_2023_isolated_aux.csv` | 고립 시트 중 19~34세 Q21 (보조, 은둔 기준 미적용) | 생성 (Git 제외) |
| 데이터 | `data/processed/services_master.meta.json` | 생성 시각·입력 해시·코드 커밋 | 생성 (Git 제외) |
| 검증 기록 | `logs/validation/raw_hashes_step00.json` | 원자료 SHA-256 | 생성 |
| 표 | `outputs/00_data_preparation/tables/*.csv` | 병합 점검, 정제 점검, 라벨 값 분포, 은둔 정의 흐름, 보조금24 품질 | 생성 |
| 그림 | 해당 없음 | 전처리 단계라 분포 그림은 STEP 01에서 만든다 | 해당 없음 |
| HTML | `outputs/00_data_preparation/summary/00_data_preparation_summary_final.html` | 단계 요약 | 생성 |

## 구성
PART A 입력 확인 → PART B SSIS 병합·정제 → PART C 규칙 라벨 후보 → PART D 은둔 수요 테이블 → PART E 보조금24 품질 → 저장·HTML → 최종 완료 확인

## 수행 범위
포함: 로딩, 1:1 병합, 정제, (C)(D)(E)(F)의 field·rule 후보 값과 근거 문장, 수요 테이블. 제외: LLM 라벨, 사람 검수, Q1 본 분석, (G) 판정(빈칸 유지).
모든 규칙 라벨은 **후보**이며 정밀도는 아직 측정하지 않았다.

## 전체 수행 흐름
1. 입력 확인: 원자료 해시를 기록하고 보조금24 수집 로그 해시와 대조한다.
2. 처리: SSIS 목록·상세를 `servId`로 1:1 병합(기대 5,288행)하고 텍스트·중첩 리스트를 정제한다.
3. 라벨 후보: 구조화 필드 → 키워드 규칙 순서로 (C)(D)(E)(F) 값을 채우고 근거 문장을 남긴다.
4. 수요: 2024 청년삶 은둔 정의를 2023 조사에 적용해 4,448명을 재현한다.
5. 저장·검증: CSV·검증 기록·HTML을 만들고 `tests/`로 행 수·키·재현·해시·값 범위를 검사한다.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("ROOT =", ROOT.name, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| python", sys.version.split()[0])

ROOT = repo | pandas 2.3.3 | matplotlib 3.11.2 | python 3.13.18


# PART A. 입력 확인

## 1단계. 원자료 해시

**목적:** `data/raw`는 읽기 전용이다. 처리 전 해시를 기록해 테스트(`tests/test_step00_preprocess.py::test_raw_unchanged`)의 기준으로 삼는다. 보조금24는 수집 로그(`_collection.json`)의 해시와도 대조한다.

In [2]:
from src.preprocess import raw_manifest
hashes = raw_manifest.write_manifest()
coll = json.loads((ROOT / "data/raw/gov24/_collection.json").read_text(encoding="utf-8"))
logged = {f"data/raw/gov24/{f['name']}": (f["sha256"], f["bytes"]) for f in coll["files"]}
hash_check = pd.DataFrame([
    {"file": k, "bytes_now": (ROOT / k).stat().st_size, "sha256_now": v[:16],
     "bytes_logged": logged.get(k, (None, None))[1], "sha256_logged": (logged.get(k, (None, None))[0] or "")[:16],
     "match_collection_log": (logged[k][0] == v) if k in logged else None}
    for k, v in hashes.items()])
display(hash_check)

,file,bytes_now,sha256_now,bytes_logged,sha256_logged,match_collection_log
0,data/raw/ssis/central_list.csv,219849,8830f713b4c66641,NaN,,None
1,data/raw/ssis/central_detail_full.csv,1760276,9235475da0ca01b8,NaN,,None
2,data/raw/ssis/local_list.csv,2408850,f9e5e9cbc065e838,NaN,,None
3,data/raw/ssis/local_detail_full.csv,7360214,ba2dc90db51d6cb5,NaN,,None
4,data/raw/gov24/gov24_serviceList.csv,12563633,10e323344eba8bfb,12633115.0,0f4961d48cb7e482,False
5,data/raw/gov24/gov24_serviceDetail.csv,14450191,7f0ac4fe0f345713,14559830.0,5f020d6dd4a77242,False
6,data/raw/gov24/gov24_supportConditions.csv,1385863,c9b974ce0399fba0,1385863.0,c9b974ce0399fba0,True
7,data/raw/isolated_youth_survey_2023/보건복지부_고립_은둔 청년 실태 조사_20230904.xlsx,7678458,2c28ea6702702cb8,NaN,,None


In [3]:
mis = hash_check[hash_check["match_collection_log"] == False]
display(Markdown(f"원인 확인 결과는 `logs/validation/gov24_hash_check_2026-11-02.json`에 있다. **관찰:** 원자료 {len(hash_check)}개 파일의 해시를 `logs/validation/raw_hashes_step00.json`에 기록했다. "
                 f"보조금24 3개 중 수집 로그와 해시가 다른 파일: {', '.join(mis['file'].str.split('/').str[-1]) or '없음'} "
                 f"(바이트 수도 다름)."))

원인 확인 결과는 `logs/validation/gov24_hash_check_2026-11-02.json`에 있다. **관찰:** 원자료 8개 파일의 해시를 `logs/validation/raw_hashes_step00.json`에 기록했다. 보조금24 3개 중 수집 로그와 해시가 다른 파일: gov24_serviceList.csv, gov24_serviceDetail.csv (바이트 수도 다름).

**해석과 한계:** SSIS·2023 조사 파일은 비교할 수집 로그 해시가 없어, 이번 기록이 이후 불변 검사의 기준이 된다. 보조금24 목록·상세 파일의 불일치는 값 변경이 아니라 수집 뒤 셀 안 줄바꿈이 CRLF→LF로 바뀐 것이다(목록은 10월 31일 파생 파일로 원본을 재구성해 수집 로그 해시와 일치 확인, 상세는 7개 컬럼과 바이트 수로 확인). 원본은 덮어쓰지 않았고, 정제 단계에서 줄바꿈을 통일하므로 분석 값에는 영향이 없다.

# PART B. SSIS 병합·정제

## 2단계. 목록·상세 1:1 병합

**목적:** 확정 결정 1) — `servId`로 목록·상세를 1:1 병합해 5,288행, 키 중복 0을 확인한다. 기대와 다르면 `build_base`가 예외로 중단한다.

In [4]:
from src.preprocess import ssis_master as sm
raw = {n: sm.read_raw(n) for n in ["central_list", "central_detail_full", "local_list", "local_detail_full"]}
merged = sm.load_merged()
merge_check = pd.DataFrame([
    {"source": s, "list_rows": len(raw[f"{s}_list"]), "detail_rows": len(raw[f"{s}_detail_full"]),
     "merged_rows": len(merged[s]), "servId_dup": int(merged[s]["servId"].duplicated().sum()),
     "list_only": len(set(raw[f"{s}_list"].servId) - set(raw[f"{s}_detail_full"].servId)),
     "detail_only": len(set(raw[f"{s}_detail_full"].servId) - set(raw[f"{s}_list"].servId))}
    for s in ["central", "local"]])
merge_check.loc[len(merge_check)] = ["total", merge_check.list_rows.sum(), merge_check.detail_rows.sum(),
                                     merge_check.merged_rows.sum(), merge_check.servId_dup.sum(), 0, 0]
merge_check["cross_source_dup"] = [None, None, len(set(merged["central"].servId) & set(merged["local"].servId))]
display(merge_check)

,source,list_rows,detail_rows,merged_rows,servId_dup,list_only,detail_only,cross_source_dup
0,central,461,461,461,0,0,0,NaN
1,local,4827,4827,4827,0,0,0,NaN
2,total,5288,5288,5288,0,0,0,0.0


**관찰:** 위 표의 `total` 행이 병합 결과다. 중앙·지자체 사이에 겹치는 `servId`는 `cross_source_dup`에 표시했다.

### Decision Box 1 — 태그·제공유형의 기준 파일
- **문제:** 중앙은 목록과 상세에 같은 태그·제공유형 컬럼이 있고 값이 일부 다르다(COLUMNS.md). 어느 쪽을 쓸지 정해야 한다.
- **선택:** 태그(`life_tags`, `target_tags`, `theme_tags`)와 `provision_type`은 **목록 파일** 기준. 코드: `src/preprocess/ssis_master.py::build_base`.
- **근거:** 아래 셀의 비교 — 생애주기·대상 태그는 집합이 같고 순서만 다르며, 중앙 관심주제는 목록 결측이 더 적고, 중앙 상세 `srvPvsnNm`은 결측이 있다.
- **결론에 미친 영향/한계:** 중앙 관심주제 집합이 다른 행은 목록 값을 따른다. 상세 값이 더 정확한지는 확인하지 않았다.
- **후속 검증:** STEP 01 태그 품질 점검에서 결측률을 다시 본다.

In [5]:
def tagset(s):
    return s.fillna("").map(lambda v: frozenset(t for t in pd.Series([v]).str.split(r"\s*,\s*").iloc[0] if t))
rows = []
for s, pairs in {"central": ["lifeArray", "trgterIndvdlArray", "intrsThemaArray", "srvPvsnNm"],
                 "local": ["lifeNmArray", "trgterIndvdlNmArray", "intrsThemaNmArray", "srvPvsnNm"]}.items():
    m = merged[s]
    for c in pairs:
        rows.append({"source": s, "column": c, "list_missing": int(m[c].isna().sum()), "detail_missing": int(m[f"{c}_det"].isna().sum()),
                     "set_differs": int((tagset(m[c]) != tagset(m[f"{c}_det"])).sum()),
                     "string_differs": int((m[c].fillna("") != m[f"{c}_det"].fillna("")).sum())})
tag_source_check = pd.DataFrame(rows)
display(tag_source_check)

,source,column,list_missing,detail_missing,set_differs,string_differs
0,central,lifeArray,165,165,0,209
1,central,trgterIndvdlArray,181,181,0,90
2,central,intrsThemaArray,12,56,44,255
3,central,srvPvsnNm,0,45,45,80
4,local,lifeNmArray,1469,1469,0,1146
5,local,trgterIndvdlNmArray,2779,2779,0,259
6,local,intrsThemaNmArray,1220,1220,0,908
7,local,srvPvsnNm,1,1,0,125


## 3단계. 텍스트·중첩 리스트 정제

**목적:** `\r` → 줄바꿈, `html.unescape`, 공백 정리. 중첩 리스트(문의처·홈페이지·근거법령·서식)는 평탄화 컬럼(1개)과 JSON 컬럼(2개 이상)을 하나의 JSON 리스트 `{contact,homepage,law,form}_json`으로 합친다. 중앙 `applmetList`에서는 '신청기관연락처목록' 항목만 골라 `apply_text`로 쓴다.

In [6]:
base = sm.build_base(merged)
text_cols = ["summary_text", "target_text", "criteria_text", "benefit_text", "apply_text_raw", "overview_text"]
clean_check = []
for c in ["summary_text", "target_text", "criteria_text", "benefit_text"]:
    rawc = base[f"{c}_raw"].fillna("")
    clean_check.append({"column": c, "cells_with_CR_raw": int(rawc.str.contains("\r").sum()),
                        "cells_with_entity_raw": int(rawc.str.contains(r"&#?\w+;").sum()),
                        "cells_with_CR_clean": int(base[c].fillna("").str.contains("\r").sum()),
                        "cells_with_entity_clean": int(base[c].fillna("").str.contains(r"&#\d+;").sum()),
                        "missing_clean": int(base[c].isna().sum())})
for g in ["contact", "homepage", "law", "form"]:
    n = base[f"{g}_json"].map(lambda v: len(json.loads(v)))
    clean_check.append({"column": f"{g}_json", "services_with_items": int((n > 0).sum()), "items_total": int(n.sum()),
                        "services_without_items": int((n == 0).sum())})
clean_check.append({"column": "apply_text (central, 신청기관 항목)", "missing_clean": int(base.loc[base.source == "central", "apply_text"].isna().sum())})
clean_check = pd.DataFrame(clean_check)
display(clean_check)

,column,cells_with_CR_raw,cells_with_entity_raw,cells_with_CR_clean,cells_with_entity_clean,missing_clean,services_with_items,items_total,services_without_items
0,summary_text,0.0,3.0,0.0,0.0,0.0,NaN,NaN,NaN
1,target_text,475.0,36.0,0.0,0.0,0.0,NaN,NaN,NaN
2,criteria_text,377.0,43.0,0.0,0.0,0.0,NaN,NaN,NaN
3,benefit_text,698.0,34.0,0.0,0.0,0.0,NaN,NaN,NaN
4,contact_json,NaN,NaN,NaN,NaN,NaN,5279.0,6529.0,9.0
5,homepage_json,NaN,NaN,NaN,NaN,NaN,993.0,1292.0,4295.0
6,law_json,NaN,NaN,NaN,NaN,NaN,5286.0,5644.0,2.0
7,form_json,NaN,NaN,NaN,NaN,NaN,4647.0,5748.0,641.0
8,"apply_text (central, 신청기관 항목)",NaN,NaN,NaN,NaN,65.0,NaN,NaN,NaN


**관찰:** 정제 후 `\r`과 숫자형 HTML 엔티티가 남은 셀 수는 `*_clean` 열에서 확인한다. 중첩 그룹은 두 형태를 합친 뒤의 항목 보유 서비스 수다(MISSING.md의 '실질결측률'과 대응).

**해석과 한계:** 원문 보존을 위해 `*_raw` 컬럼을 함께 남긴다. 정제는 표기만 바꾸며 문장 내용을 줄이지 않는다.

# PART C. 규칙 라벨 후보 (C)(D)(E)(F)

## 4단계. services_master 생성

**목적:** 각 라벨 `<속성>`에 `_evidence`(근거 원문 문장), `_method`(field/rule), `_review`(검수 필요)를 붙인다. 근거 문장이 없으면 `N`, 해당 없음은 `NA`. 사전·검색식: `src/preprocess/keywords.py`.

### Decision Box 2 — 라벨 값을 정하는 순서와 '후보' 표시
- **문제:** 같은 속성에 구조화 필드와 원문 키워드가 서로 다른 값을 줄 수 있다. 원문 키워드는 배경 설명(사업 목적)에도 걸린다.
- **선택:** ① 구조화 필드(`onapPsbltYn`, `aplyMtdNm`, `srvPvsnNm`, `sprtCycNm`) → ② 키워드 규칙. 원문 키워드로 정한 (D)(E) 값과 need_area 후보는 모두 `review=True`. 필드에서 나온 값은 `review=False`(조건부 예외는 코드 주석).
- **근거:** 규칙 근거 표본을 직접 읽었을 때 '면접정장 대여'가 면접으로, 사업 배경의 '근무'가 출근형으로 잡히는 오탐이 있었다(일부는 제외어로 막음). 정밀도를 측정하기 전에는 확정 값으로 쓸 수 없다.
- **결론에 미친 영향/한계:** review 비율이 높게 나오며, 이는 검수량을 뜻하지 품질 판정이 아니다. 규칙 값의 정확도는 미검증.
- **후속 검증:** STEP 02에서 표본 사람 검수로 규칙별 정밀도를 측정한다.

### Decision Box 3 — 접근성 (D) 세부 규칙
- **문제:** 지시서의 기준표를 원문에 적용하는 방법이 여러 가지다.
- **선택:**
  - `apply_barrier`: 가능한 신청 경로 중 **가장 쉬운 경로**(최솟값). `onapPsbltYn=Y`·`aplyMtdNm`의 인터넷/모바일/E-mail → 0, 전화·우편·팩스 → 1, 방문 → 2, 방문 + 원문의 면접 → 3. '별도 신청사항 없음'·자동 지급은 0 + review. '다수 서류'는 규칙으로 판정하지 않았다.
  - `online_bokjiro`는 중앙 `onapPsbltYn`만 담고 의미를 '복지로 온라인 신청 가능'으로 한정한다. 지자체는 해당 필드가 없어 빈칸이다.
  - `use_barrier`: 제공유형이 현금·융자·감면·지역화폐·바우처로만 이루어지면 0(field). 그 밖에는 원문 키워드 중 **가장 높은 값**(이용은 구성요소를 모두 거쳐야 한다고 봄). 시설입소는 기준표에 대응이 없어 N + review.
  - `repeat_burden`: `sprtCycNm=1회성` → 0. 현금형의 월·분기 등 지급 주기는 참여 횟수가 아니므로 N + review. 그 밖에는 '매주·매월·N개월(≥2)' → 2, 'N회·N주' → 1(8회 초과·6주 초과는 2).
  - `remote_complete`: 신청 비대면 = apply 0·1(전화·우편 포함), 이용 비대면 = use 0. 어느 한쪽이 N이면 N.
- **근거:** 기준표 문구(0 온라인 완결 / 1 전화·우편·대리 / 2 방문 1회 / 3 방문+면접·다수 서류)와 추천 목적(사용자가 고를 수 있는 가장 쉬운 신청 경로).
- **결론에 미친 영향/한계:** 최솟값 규칙은 서류 제출용 방문이 따로 필요한 경우를 놓칠 수 있다. 숫자 경계(8회·6주)는 임의 기준이며 팀 확인 필요.
- **후속 검증:** STEP 02 검수 표본에서 apply_barrier 0 판정의 실제 온라인 완결 여부를 확인.

### Decision Box 4 — 추천 요소 (E) 세부 규칙
- **선택:** `outing_burden`은 지시서 변환식 그대로(한쪽만 N이면 다른 쪽 값, 이때 review). `cost`: 융자 포함 → 2(field), 현금지급·지역화폐만 → NA(field), 그 밖에는 원문(무료·전액 지원 → 0, 본인부담·자부담 → 1, 상환·대출 → 2). `contact_burden`: 집에서 이용하는 제공유형만 → 0(field), 원문의 면접·집단 → 2, 상담·멘토·개별 → 1. `job_relevance`: 직접(취업알선·일경험·직업훈련·구직) → 2, 간접(자격·진로·자산형성) → 1, **0(무관)은 채우지 않음**.
- **근거:** '근거 문장이 없으면 N, N과 0을 혼용하지 않는다'. 직업 관련 키워드가 없다는 사실은 무관하다는 근거 문장이 아니다.
- **결론에 미친 영향/한계:** `job_relevance`는 대부분 N으로 남는다. 0은 LLM 또는 사람이 근거 문장을 찾아야 채울 수 있다.
- **후속 검증:** STEP 01 PART 5의 N 비율로 항목별 라벨 방법을 판정.

### Decision Box 5 — 자격 (C) 연령 추출
- **문제:** 원문의 'N세'는 신청자 본인뿐 아니라 자녀·배우자·가구원 연령도 가리킨다. 출생연도 표기도 있다.
- **선택:** `N~M세`, `N세 이상 M세 이하`, `N세 이상/이하/미만` 패턴을 서비스명·지원대상·선정기준·요약에서 찾고, 앞 8자에 자녀·배우자·부모·가구원 등이 있거나 뒤 14자에 '자녀', '~이 있는', '~를 둔', '양육', '구성원' 등이 있으면 제외. 여러 범위는 최솟값~최댓값의 합집합으로 두고 review. 한쪽 경계가 없으면 `NA`(예: 65세 이상 → age_max=NA). 출생연도 표기는 다루지 않음(→ N).
  - `elig_youth`: 범위가 19~34세와 겹치면 '예', 범위가 확인됐는데 겹치지 않으면 '아니오', 연령 정보가 없으면 N. 하한 = '예', 상한 = '예' + N.
  - 청년 근거 플래그: `youth_tag`(생애주기 태그 '청년'), `youth_text`(서비스명·요약·대상·기준의 '청년'), `youth_age`(위 범위 겹침). 셋을 섞지 않는다.
- **근거:** WLF00006301 원문(가족돌봄 13~34세, 고립은둔 19~34세)에서 '가족돌봄청년'의 '가족'이 가족 구성원 제외 규칙에 걸려 범위가 잘못 줄어든 것을 확인해 뒤 문맥 제외어에서 '가족'을 뺐다.
- **결론에 미친 영향/한계:** 합집합은 AND 조건(예: '19세 이상' + '39세 이하')과 OR 조건(집단별 범위)을 구분하지 못한다. 정확도 미검증.
- **후속 검증:** STEP 01에서 태그·원문·연령 플래그의 불일치(태그는 청년인데 연령은 아니오 등)를 확인.

In [7]:
master = sm.build_master()
label_attrs = ["elig_youth", "elig_income", "isolation_explicit", "target_group", "apply_barrier", "use_barrier",
               "repeat_burden", "remote_complete", "outing_burden", "duration", "cost", "contact_burden", "job_relevance",
               "use_outing", "outing_need"]
rows = []
for a in label_attrs:
    for (src, val), n in master.groupby(["source", a]).size().items():
        sub = master[(master.source == src) & (master[a] == val)]
        rows.append({"attribute": a, "source": src, "value": val, "n": int(n),
                     "n_field": int((sub[f"{a}_method"] == "field").sum()), "n_rule": int((sub[f"{a}_method"] == "rule").sum()),
                     "n_review": int(sub[f"{a}_review"].astype(bool).sum())})
label_value_counts = pd.DataFrame(rows)
print(master.shape)
display(label_value_counts.pivot_table(index=["attribute", "value"], columns="source", values="n", fill_value=0))

(5288, 153)


source                    central   local
attribute          value                 
apply_barrier      0         64.0  1281.0
                   1          2.0   518.0
                   2        331.0  2332.0
                   3          0.0     4.0
                   N         64.0   692.0
contact_burden     0        242.0  3276.0
                   1         55.0    97.0
                   2         28.0    54.0
                   N        136.0  1400.0
cost               0         18.0   176.0
                   1         51.0   215.0
                   2         33.0    61.0
                   N        193.0  1608.0
                   NA       166.0  2767.0
duration           0         80.0  1656.0
                   2         21.0   140.0
                   N        360.0  3031.0
elig_income        N        269.0  3095.0
                   없음         4.0    19.0
                   있음       188.0  1713.0
elig_youth         N        303.0  3250.0
                   아니오       74.0  1000.0
                   예         84.0   577.0
isolation_explicit 아니오      459.0  4810.0
                   예          2.0    17.0
job_relevance      1         22.0   107.0
                   2         31.0   147.0
                   N        408.0  4573.0
outing_burden      0         75.0  1498.0
                   1        254.0  2707.0
                   2        112.0   343.0
                   N         20.0   279.0
outing_need        0         32.0  1057.0
                   1         24.0   271.0
                   2        227.0  2035.0
                   3          1.0     1.0
                   N        177.0  1463.0
remote_complete    0         81.0   118.0
                   1        190.0  2083.0
                   2         54.0  1201.0
                   N        136.0  1425.0
repeat_burden      0         80.0  1656.0
                   2         21.0   140.0
                   N        360.0  3031.0
target_group       가족돌봄       4.0     7.0
                   고립·은둔      1.0    17.0
                   공통         1.0     0.0
                   불명       455.0  4803.0
use_barrier        0        255.0  3438.0
                   1          3.0    32.0
                   2         88.0   296.0
                   3         23.0    42.0
                   N         92.0  1019.0
use_outing         0        224.0  3030.0
                   1         44.0   579.0
                   2         42.0   182.0
                   3          1.0     1.0
                   N        150.0  1035.0

In [8]:
ex = master.set_index("servId").loc["WLF00006301", ["servNm", "age_min", "age_max", "age_evidence", "elig_youth", "elig_income",
       "elig_income_review", "isolation_explicit", "target_group", "target_group_evidence", "target_group_review", "need_area_q21", "app_domain"]]
display(ex.to_frame("WLF00006301"))
iso = master[master.isolation_explicit == "예"]
display(Markdown(f"**관찰:** 마스터 {len(master):,}행 × {master.shape[1]}열. 고립·은둔을 명시한 서비스 {len(iso)}건 중 "
                 f"target_group 검수 대상 {int(iso.target_group_review.astype(bool).sum())}건(청년 근거가 없거나 여러 집단이 함께 적힘). "
                 f"WLF00006301은 target_group='{ex.target_group}', review={bool(ex.target_group_review)}."))

,WLF00006301
servNm,"위기청년(가족돌봄, 고립은둔) 전담 지원사업"
age_min,13
age_max,34
age_evidence,"가족돌봄청(소)년(13세~34세)으로, 가족돌봄청년 세부선정 기준은 아래와 같습니다. | 고립은둔청년(19세~34세)으로, 세부 선정기준은 아래와 같습니다."
elig_youth,예
elig_income,있음
elig_income_review,True
isolation_explicit,예
target_group,공통
target_group_evidence,"위기청년(가족돌봄, 고립은둔) 전담 지원사업 | 위기청년(가족돌봄, 고립은둔) 전담 지원사업"


**관찰:** 마스터 5,288행 × 153열. 고립·은둔을 명시한 서비스 19건 중 target_group 검수 대상 11건(청년 근거가 없거나 여러 집단이 함께 적힘). WLF00006301은 target_group='공통', review=True.

## 4-1단계. 라벨 기준 개정 (2026-11-02, v2)

**목적:** 팀 구현 계획 개정과 팀 검토를 반영한다. 기존 컬럼의 의미를 바꾸지 않도록 다음처럼 나눴다.
- `apply_barrier`: 개정(개정 전 값은 `apply_barrier_v1`). 신청 경로는 가장 쉬운 값, '심사'는 3에서 제외, **필수 면접**만 3으로 올림(`apply_interview`: 현장/비대면/불명). 경로 키워드는 같은 문장에 신청·접수·제출이 있고 부정 표현이 없을 때만 근거로 쓴다. **명시적 신청 불필요**만 0 + 검수, '직접 지원'만 있거나 정보가 없으면 N. 추천·직권은 경로대로 두고 검수.
- `use_barrier`·`outing_burden`: **유지.** 새 개념은 `use_outing`(서비스 이용 외출 조건 0~3)과 `outing_need`(새 외출필요도 0~3)로 따로 만든다.
- `outing_need` = max(신청 환산 {0→0, 1→0, 2→1, 3→2}, use_outing). 비대면 면접은 면접 전 경로 값(`apply_channel`)으로 환산. 한쪽이 N이면 N, 단 아는 쪽이 3이면 3. 확인된 하한은 `outing_need_min`.
- `remote_complete`: 개정된 apply_barrier로 다시 계산(개정 전 값은 `remote_complete_v1`).
- `app_domain`: 대응표 v2의 **확정 대응만**. 미확정 대응은 `app_domain_pending`(개정 전 값은 `app_domain_v1`).
- 규칙 값은 모두 `src/preprocess/keywords.py`의 v2 구역에 있다. use_outing·outing_need는 F7 외출 단계를 차용했지만 사용자 외출 상태와 1:1 대응하지 않으므로 추천 자격 기준으로 쓰지 않는다.

In [9]:
pairs = [("apply_barrier", "apply_barrier_v1"), ("remote_complete", "remote_complete_v1"), ("app_domain", "app_domain_v1"),
         ("use_outing", "use_barrier"), ("outing_need", "outing_burden")]
rows = []
for new, old in pairs:
    t = master.groupby([old, new]).size().rename("n").reset_index().rename(columns={old: "before", new: "after"})
    t.insert(0, "attribute", new); t.insert(1, "compared_with", old)
    t["changed"] = t.before != t.after
    t["note"] = "개정(같은 개념)" if old.endswith("_v1") else "새 컬럼(다른 개념, 참고 비교)"
    rows.append(t)
label_revision_diff = pd.concat(rows, ignore_index=True)
rev_summary = label_revision_diff.groupby(["attribute", "compared_with", "note"]).apply(
    lambda g: pd.Series({"n_changed": int(g.loc[g.changed, "n"].sum()), "n_total": int(g.n.sum())}), include_groups=False).reset_index()
display(rev_summary)
display(label_revision_diff[label_revision_diff.changed & label_revision_diff.attribute.eq("apply_barrier")])
dist = pd.DataFrame({c: master[c].value_counts() for c in ["apply_barrier_v1", "apply_barrier", "use_barrier", "use_outing",
                                                         "outing_burden", "outing_need", "outing_need_min"]}).fillna(0).astype(int)
display(dist)
display(Markdown(f"**관찰:** apply_barrier 개정으로 {int(rev_summary.loc[rev_summary.attribute.eq('apply_barrier'), 'n_changed'].iloc[0]):,}건이 바뀌었다. "
                 f"필수 면접 {int(master.apply_interview.ne('없음').sum())}건(형태: {master.apply_interview.value_counts().drop('없음', errors='ignore').to_dict()}). "
                 f"outing_need N {int(master.outing_need.eq('N').sum()):,}건 중 하한이 확인된 건 {int((master.outing_need.eq('N') & master.outing_need_min.ne('N')).sum()):,}건. "
                 f"use_outing 3(정기 외출)은 {int(master.use_outing.eq('3').sum())}건으로 규칙으로는 드물게만 잡힌다."))
from src.preprocess import keywords as K
map_v2 = pd.DataFrame(K.APP_DOMAIN_MAP_V2, columns=["q21_area", "app_domain", "status", "condition", "note"])
map_v2["app_domain"] = map_v2.app_domain.fillna("(대응 없음)")
map_v2["q21_label"] = map_v2.q21_area.map(K.Q21_LABELS)
display(map_v2)
display(Markdown(f"**관찰:** app_domain 확정값이 있는 서비스 {int(master.app_domain.ne('N').sum()):,}건, 미확정 대응 후보가 있는 서비스 {int(master.app_domain_pending.notna().sum()):,}건."))

,attribute,compared_with,note,n_changed,n_total
0,app_domain,app_domain_v1,개정(같은 개념),3214,5288
1,apply_barrier,apply_barrier_v1,개정(같은 개념),133,5288
2,outing_need,outing_burden,"새 컬럼(다른 개념, 참고 비교)",3993,5288
3,remote_complete,remote_complete_v1,개정(같은 개념),81,5288
4,use_outing,use_barrier,"새 컬럼(다른 개념, 참고 비교)",1110,5288


,attribute,compared_with,before,after,n,changed,note
1,apply_barrier,apply_barrier_v1,0,1,2,True,개정(같은 개념)
2,apply_barrier,apply_barrier_v1,0,2,18,True,개정(같은 개념)
3,apply_barrier,apply_barrier_v1,0,3,2,True,개정(같은 개념)
5,apply_barrier,apply_barrier_v1,1,2,18,True,개정(같은 개념)
6,apply_barrier,apply_barrier_v1,1,N,30,True,개정(같은 개념)
7,apply_barrier,apply_barrier_v1,2,0,1,True,개정(같은 개념)
8,apply_barrier,apply_barrier_v1,2,1,51,True,개정(같은 개념)
10,apply_barrier,apply_barrier_v1,2,N,4,True,개정(같은 개념)
11,apply_barrier,apply_barrier_v1,3,2,5,True,개정(같은 개념)
13,apply_barrier,apply_barrier_v1,N,0,2,True,개정(같은 개념)


,apply_barrier_v1,apply_barrier,use_barrier,use_outing,outing_burden,outing_need,outing_need_min
0,1364,1345,3693,3254,1573,1089,1837
1,515,520,35,623,2961,295,343
2,2678,2663,384,224,455,2262,2803
3,7,4,65,2,0,2,2
N,724,756,1111,1185,299,1640,303


**관찰:** apply_barrier 개정으로 133건이 바뀌었다. 필수 면접 4건(형태: {'불명': 3, '현장': 1}). outing_need N 1,640건 중 하한이 확인된 건 1,337건. use_outing 3(정기 외출)은 2건으로 규칙으로는 드물게만 잡힌다.

,q21_area,app_domain,status,condition,note,q21_label
0,q21_11,주거,확정,None,,주거 (Q21에 대응 없음)
1,q21_05,건강,확정,None,,전문가에 의한 상담·치료
2,q21_02,건강,미확정,None,관계 요소도 있음,일상생활 회복 지원
3,q21_07,교육 및 훈련,확정,None,,교육·학습 지원
4,q21_08,미취업,확정,None,,취업(정보)·일 경험 지원
5,q21_10,경제(생활비),확정,None,,경제적 지원
6,q21_10,노동,미확정,labor,근로자 지원·자산형성 근거가 있을 때만,경제적 지원
7,q21_01,관계 및 참여,미확정,None,공간 제공 자체가 관계 영역인지 불분명,눈치 보지 않고 들러서 머물 수 있는 공간
8,q21_04,관계 및 참여,확정,None,,같이 하는 취미·문화·체육활동 지원
9,q21_06,관계 및 참여,확정,None,,"또래, 멘토 등에 의한 상담·지지"


**관찰:** app_domain 확정값이 있는 서비스 1,720건, 미확정 대응 후보가 있는 서비스 1,576건.

### Decision Box 6 — v2 개정에서 남은 경계 사례
- 추천·직권(기관 주도 선정): 원문에 사용자 방문·관공서 신청이 있으면 경로 값(대개 2) + 검수, 없으면 '방문' 분류값만으로 2를 주지 않고 N + 검수.
- 필수 면접 형태 미기재(`apply_interview=불명`): 신청 쪽 외출을 추정하지 않는다. outing_need는 이용 쪽이 3일 때만 값이 있고, 하한(outing_need_min)에는 신청 경로 값만 반영한다.
- 대리 신청(apply_barrier 1)은 대리인이 방문하므로 remote_complete에서 비대면 신청으로 세지 않는다.
- 규칙 값 검토(2026-11-02, AI 검토·비확정): 원문 근거가 없는 명백한 오분류만 `src/preprocess/review_overrides.py`에서 N으로 되돌린다. 사례와 사유는 `outputs/00_data_preparation/review/`.
- 시설입소: use_outing 정의에 없어 N + 검수.
- use_outing 3 규칙은 지원 항목 나열 문장('직업훈련 등 제공')에도 걸린다. 검토에서 나열 오탐은 N으로 되돌렸고, 남은 규칙 값도 후보이며 STEP 02 사람 판정(`h_use_outing`)으로 확인한다.

## 4-2단계. v3 보완 (2026-11-04): 청년 범위·연령 경계·영역 근거

**목적:** Q1 공급 후보 모집단과 LLM 추천이 같은 서비스 원본을 쓰면서 청년 대상 여부의 불확실성을 따로 다루도록 한다. 기존 컬럼은 개정 전 값을 `*_v1`로 남기고, 새 컬럼은 마스터 맨 뒤에 붙인다.

### Decision Box 7 — v3에서 고친 것과 추가한 것
- **오류 수정 1 (연령 합집합):** v1은 여러 연령 범위의 합집합에서 열린 경계를 버렸다('18세 이상' + '18~20세' → 18~20, '65세 이상' + '65세 미만' → 65~64). v3는 열린 경계를 `NA`로 두고, 청년 연령 겹침(`youth_age`)은 개별 범위로 판정한다(6~18세 또는 70세 이상은 겹치지 않음).
- **오류 수정 2 (영역 근거):** 명세 5절은 '주거급여 수급자'처럼 자격 조건에만 나온 표현을 영역 근거로 쓰지 않는다고 정했지만, v1 규칙은 요약·지원내용 문장 안의 '생계·의료급여 수급자'에 반응해 ⑩·⑪을 붙였다. v3는 이 표현만 가린 뒤 판정한다. 영역을 새로 만들지 않고 지우기만 한다.
- **오류 수정 3 (청년 언급):** '신청년도'(신청 + 년도)가 '청년'으로 잡혔다. `(?<!신)청년`으로 바꿨다.
- **추가:** `youth_scope`(A/B/U/C)와 근거 종류 `youth_scope_basis`, 특정 집단 조건 `target_condition`, ⑩ 근거 종류 `q21_10_basis`. 정의는 `docs/services_master_spec.md`.
- **유지:** 사용자 적격성은 마스터에서 판정하지 않는다(`src/recommend/eligibility.py`). 외출 관련 라벨(v2)은 바꾸지 않았다.
- **대안과 기각 이유:** '청년' 글자 유무만으로 A/C를 나누는 방식은 기초생활보장·주거급여처럼 청년 전용이 아닌 서비스를 빼고, 근거가 없는 서비스를 '청년 비해당'으로 잘못 확정한다. 그래서 근거가 없거나 충돌하면 U로 둔다.

In [10]:
rev_rows = []
for new, old in [("need_area_q21", "need_area_q21_v1"), ("elig_youth", "elig_youth_v1"), ("age_min", "age_min_v1"), ("age_max", "age_max_v1")]:
    ch = master[new].fillna("") != master[old].fillna("")
    rev_rows.append({"attribute": new, "changed_services": int(ch.sum()),
                     "examples": ", ".join(master.loc[ch, "servId"].head(5))})
label_revision_v3 = pd.DataFrame(rev_rows)
display(label_revision_v3)
ey = pd.crosstab(master.elig_youth_v1, master.elig_youth).rename_axis(index="v1", columns="v3")
display(ey)
youth_scope_counts = (master.groupby(["youth_scope", "youth_scope_basis", "source"]).size().unstack("source", fill_value=0)
                      .assign(total=lambda x: x.sum(axis=1)).reset_index())
display(youth_scope_counts)
v1a = master.need_area_q21_v1.str.split("|").explode().value_counts().rename("v1")
v3a = master.need_area_q21.str.split("|").explode().value_counts().rename("v3")
area_v3 = pd.concat([v1a, v3a], axis=1).fillna(0).astype(int).assign(diff=lambda x: x.v3 - x.v1)
display(area_v3[area_v3["diff"].ne(0)])
(ROOT / "outputs/00_data_preparation/tables").mkdir(parents=True, exist_ok=True)
label_revision_v3.to_csv(ROOT / "outputs/00_data_preparation/tables/label_revision_v3_diff.csv", index=False, encoding="utf-8-sig")
youth_scope_counts.to_csv(ROOT / "outputs/00_data_preparation/tables/youth_scope_counts.csv", index=False, encoding="utf-8-sig")
display(Markdown(f"**관찰:** v3에서 영역이 바뀐 서비스 {int(rev_rows[0]['changed_services'])}건, 연령 겹침 판정이 바뀐 서비스 {int(rev_rows[1]['changed_services'])}건. "
                 f"청년 범위는 A {int((master.youth_scope=='A').sum()):,} / B {int((master.youth_scope=='B').sum()):,} / "
                 f"U {int((master.youth_scope=='U').sum()):,} / C {int((master.youth_scope=='C').sum()):,}건이다. 규칙 후보이며 사람 판정 전이다."))

,attribute,changed_services,examples
0,need_area_q21,1930,"WLF00000024, WLF00000025, WLF00000026, WLF00000033, WLF00000036"
1,elig_youth,60,"WLF00001086, WLF00001087, WLF00001107, WLF00001178, WLF00003194"
2,age_min,112,"WLF00001067, WLF00001076, WLF00001086, WLF00001087, WLF00001091"
3,age_max,115,"WLF00000098, WLF00001086, WLF00001087, WLF00001091, WLF00001093"


v3,N,아니오,예
v1,,,
N,3548,0,0
아니오,3,1074,55
예,2,0,606


source,youth_scope,youth_scope_basis,central,local,total
0,A,age_band,0,17,17
1,A,name_text,9,265,274
2,B,age_overlap,77,318,395
3,B,tag_only,83,746,829
4,B,target_text_mention,6,9,15
5,B,text_mention,1,7,8
6,C,age_excludes,10,275,285
7,U,age_may_refer_to_dependent,32,273,305
8,U,age_partial,22,406,428
9,U,conflict,10,46,56


,v1,v3,diff
q21_10,3620,1746,-1874
N,1242,2230,988
q21_11,492,490,-2
q21_08,357,376,19
q21_07,281,545,264


**관찰:** v3에서 영역이 바뀐 서비스 1930건, 연령 겹침 판정이 바뀐 서비스 60건. 청년 범위는 A 291 / B 1,247 / U 3,465 / C 285건이다. 규칙 후보이며 사람 판정 전이다.

## 4-3단계. 외출 환산 v3 (2026-11-05, 잠정 라벨)

### Decision Box 8 — 방문 신청의 외출 환산
- **문제:** v2는 신청 장벽 2(방문)를 외출 1(근처 짧은 외출)로 환산했다. 같은 척도에서 이용 쪽 기관 방문은 2이고, 3차 회의 '외출 필요도 정의' 문서도 방문 신청을 '외출 필수'로 정의해 서로 맞지 않았다. 환산 하나로 2,094건이 1과 2 사이를 오간다.
- **선택:** 신청 쪽 외출 `apply_outing`(0/2/N)을 따로 두고 방문 신청을 **잠정 2**로 둔다. 근거 수준은 `apply_outing_basis`에 남긴다.
  - `visit_required_text`: 원문에 방문 필수 또는 '온라인 신청 후 방문 제출' 같은 순차 절차 → 2 (신청 장벽이 0이어도)
  - `visit_only_listed`: 방문 경로만 안내됨(대부분 신청방법 분류값 '방문') → 2 + 검수. **방문 필수로 해석하지 않는다.** 추천에서는 '방문 신청만 안내됨'으로 표시하고 제외 근거로 쓰지 않는다(`src/recommend/eligibility.py`).
  - `remote_route` 0 / `no_apply_info`·`interview_unknown` N / `interview_onsite` 2
- **보존:** v2 값은 `outing_need_v2`·`outing_need_min_v2`. 두 팀 라벨 모두 사람 검수 전이므로 이 값도 정답이 아니다. STEP 02 보충 표본 C(규칙 간 불일치)로 검증한다.

In [11]:
ct = pd.crosstab(master.outing_need_v2, master.outing_need).rename_axis(index="v2", columns="v3")
display(ct)
basis = master.apply_outing_basis.value_counts().rename_axis("apply_outing_basis").reset_index(name="n")
display(basis)
outing_v3 = ct.stack().rename("n").reset_index()
outing_v3.to_csv(ROOT / "outputs/00_data_preparation/tables/outing_v3_diff.csv", index=False, encoding="utf-8-sig")
basis.to_csv(ROOT / "outputs/00_data_preparation/tables/apply_outing_basis_counts.csv", index=False, encoding="utf-8-sig")
nb = basis.set_index("apply_outing_basis")["n"]
display(Markdown(f"**관찰:** 외출필요도가 바뀐 서비스 {int((master.outing_need != master.outing_need_v2).sum()):,}건은 모두 신청 쪽이 방문인 서비스다. "
                 f"방문 경로만 안내된 서비스 {int(nb.get('visit_only_listed', 0)):,}건, 원문상 방문 필수 {int(nb.get('visit_required_text', 0))}건. "
                 "원문에 방문 필수가 적힌 경우는 드물어, 대부분은 '다른 경로 미확인' 상태다."))

v3,0,1,2,3,N
v2,,,,,
0,1089,0,1,0,0
1,0,295,2094,0,0
2,0,0,167,0,0
3,0,0,0,2,0
N,0,0,0,0,1640


,apply_outing_basis,n
0,visit_only_listed,2663
1,remote_route,1863
2,no_apply_info,756
3,interview_unknown,3
4,visit_required_text,2
5,interview_onsite,1


**관찰:** 외출필요도가 바뀐 서비스 2,095건은 모두 신청 쪽이 방문인 서비스다. 방문 경로만 안내된 서비스 2,663건, 원문상 방문 필수 2건. 원문에 방문 필수가 적힌 경우는 드물어, 대부분은 '다른 경로 미확인' 상태다.

**해석과 한계:** 위 분포는 규칙 후보의 분포이며 서비스의 실제 속성 분포로 해석하지 않는다. 고립·은둔 '명시'에는 노인·1인가구의 사회적 고립 문구도 포함되므로, 청년 고립·은둔 서비스 수로 읽지 않는다.

## 5단계. 영역 (F) 후보와 app_domain 대응표(초안, 미확정)

**목적:** `need_area_q21`은 2023 Q21 10개 + ⑪주거 다중 라벨. 검색 대상은 서비스명·요약·지원내용(지원대상 문장은 '주거급여 수급자' 같은 자격 조건이라 제외). ⑩은 제공유형(현금지급·융자·지역화폐·감면) 필드로도 붙인다.
필수 포함: '일상복귀', '일상으로'(②). 필수 제외: '공동생활가정'(⑨에서 제외), 셰어하우스(⑪), 노숙인·청소년·피해자 쉼터(①에서 제외), 장애인 활동지원의 '일상생활 지원'(②에서 제외).

In [12]:
from src.preprocess import keywords as K
rules = pd.DataFrame([{"need_area": k, "label": K.Q21_LABELS[k], "include": v["include"], "exclude": v["exclude"]}
                      for k, v in K.NEED_AREA_RULES.items()])
display(rules)
dom_map = pd.DataFrame([{"need_area": k, "label": K.Q21_LABELS[k],
                         "app_domain": " / ".join(K.APP_DOMAIN_MAP.get(k, [])) or ("노동(근로자·자산형성 근거 시) 또는 경제(생활비)" if k == "q21_10" else "대응 없음")}
                        for k in K.Q21_LABELS])
dom_map.loc[dom_map.need_area.isin(["q21_06", "q21_07"]), "app_domain"] += " + 사회인식 및 미래설계(진로 근거 시, 팀 확인 필요)"
dom_map["status"] = "초안 · 미확정 (팀 확인)"
display(dom_map)

,need_area,label,include,exclude
0,q21_01,눈치 보지 않고 들러서 머물 수 있는 공간,쉼터|거점\s*공간|공유\s*공간|청년\s*공간|머물|휴식\s*공간|아지트|라운지|사랑방|커뮤니티\s*공간,노숙|청소년\s*쉼터|가출|피해|폭력|학대|성매매|이주여성|노인|어르신|장애|입소|휴게|육아|무료\s*급식|대출
1,q21_02,일상생활 회복 지원,"일상\s*복귀|일상으로|일상\s*회복|생활\s*습관|생활\s*리듬|일상생활\s*(지원|회복|훈련)|자기\s*돌봄|신체적.{0,3}정신적\s*회복",활동\s*지원|장애인|장애아|노인|어르신|요양|신변\s*처리|방문\s*목욕|가사\s*지원|이동\s*보조|집수리|생활습관\s*평가|퇴원
2,q21_03,혼자 하는 취미·문화·체육활동 지원,취미|(?<!입양)(?<!법)(?<!성)(?<!장례)(?<!입양 )문화(?!재|를\s*이해)|예술|체육(?!복)|(?<!심리)(?<!맞춤)운동(?!\s*(치료|재활|화|지도))|스포츠|헬스|공연|관람|여가|(...,동아리|모임|함께|단체|그룹|소모임|커뮤니티|공동체|캠프|팀|문화\s*가정|다문화|치료|재활|입학\s*준비|출전|제외\s*대상|신청\s*제외
3,q21_04,같이 하는 취미·문화·체육활동 지원,(?=.*(?:취미|(?<!입양)(?<!법)(?<!성)(?<!장례)(?<!입양 )문화(?!재|를\s*이해)|예술|체육(?!복)|(?<!심리)(?<!맞춤)운동(?!\s*(치료|재활|화|지도))|스포츠|헬스|공연...,다문화|문화\s*가정
4,q21_05,전문가에 의한 상담·치료,심리|정신\s*건강|마음\s*건강|정서\s*(지원|안정|회복)|우울|트라우마|중독|상담\s*치료|정신과,"심리\s*적\s*부담|경제\s*적\s*[,·ㆍ]?\s*심리|심리\s*[·ㆍ]?\s*사회|취사|청소(?!년)|세탁"
5,q21_06,"또래, 멘토 등에 의한 상담·지지",멘토|멘토링|또래|자조\s*모임|동료\s*지원|피어|(?<!신)선배(?!아),
6,q21_07,교육·학습 지원,(?<!체험)(?<!현장)학습|(?<!스포츠)(?<!스포츠 )강좌|강의|수강|교육\s*(과정|프로그램)|검정고시|평생\s*교육|아카데미|학업|자격증|(?<!창작)역량\s*강화,응시\s*(료|비)|소풍|수련회
7,q21_08,취업(정보)·일 경험 지원,취업|일\s*경험|구직|직업\s*훈련|인턴|일자리|채용|창업|직무,"일자리\s*창출|떠나는|여행|(주거|창업|자기\s*계발)[^.\n]{0,15}사용|주거\s*(와|및|,|·)\s*일자리|일자리\s*[,·]\s*주거|취업난\s*[·ㆍ,]\s*(높은\s*)?주거|장기화되면서\s..."
8,q21_09,공동생활(규칙적인 생활 등),공동\s*생활(?!\s*가정)|합숙|규칙적(인)?\s*생활|공동체\s*생활,공동\s*생활\s*가정|그룹홈|셰어\s*하우스|쉐어\s*하우스
9,q21_10,경제적 지원,생활비|생계비|수당|장려금|지원금|급여|현금|용돈|자산\s*형성|저축|적금,


,need_area,label,app_domain,status
0,q21_01,눈치 보지 않고 들러서 머물 수 있는 공간,관계 및 참여,초안 · 미확정 (팀 확인)
1,q21_02,일상생활 회복 지원,건강,초안 · 미확정 (팀 확인)
2,q21_03,혼자 하는 취미·문화·체육활동 지원,대응 없음,초안 · 미확정 (팀 확인)
3,q21_04,같이 하는 취미·문화·체육활동 지원,관계 및 참여,초안 · 미확정 (팀 확인)
4,q21_05,전문가에 의한 상담·치료,건강,초안 · 미확정 (팀 확인)
5,q21_06,"또래, 멘토 등에 의한 상담·지지","관계 및 참여 + 사회인식 및 미래설계(진로 근거 시, 팀 확인 필요)",초안 · 미확정 (팀 확인)
6,q21_07,교육·학습 지원,"교육 및 훈련 + 사회인식 및 미래설계(진로 근거 시, 팀 확인 필요)",초안 · 미확정 (팀 확인)
7,q21_08,취업(정보)·일 경험 지원,미취업,초안 · 미확정 (팀 확인)
8,q21_09,공동생활(규칙적인 생활 등),대응 없음,초안 · 미확정 (팀 확인)
9,q21_10,경제적 지원,노동(근로자·자산형성 근거 시) 또는 경제(생활비),초안 · 미확정 (팀 확인)


# PART D. 은둔 수요 테이블

## 6단계. 2024 청년삶 은둔 정의를 2023 조사에 적용

**목적:** 확정 결정 3). 2024 파일설계서 358(외출 빈도 ⑤~⑧)·360(은둔 생활 이유) 문항을 2023 1차조사에 대응시킨다. 2023 `sq3`의 1~4가 2024의 ⑧~⑤와 같은 보기(코딩 방향 반대)임을 두 코드북에서 확인했다. 기대값 4,448명이며, 다르면 `build_demand`가 예외로 중단한다.

In [13]:
from src.preprocess import survey_2023 as sv
from src.preprocess.xlsx_reader import read_sheet
spec_path = next(p for p in sorted((ROOT / "docs/survey_codebooks/youth_life_survey").rglob("*.xlsx")) if "2024" in p.name)
print("2024 파일설계서:", spec_path.relative_to(ROOT).as_posix())
spec = read_sheet(spec_path, "총괄", header=False)
spec_items = spec[spec[0].isin(["358", "360"])][[0, 1, 2, 3]]
spec_items.columns = ["item_no", "item", "code_2024", "label"]
cb = read_sheet(sv.survey_path(), "(코드북)1차조사", header=False)
cb[0] = cb[0].ffill()
cb_items = cb[cb[0].isin(["sq1a1", "sq3", "sq3a1"])][[0, 2, 3]]
cb_items.columns = ["var_2023", "code_2023", "label"]
display(spec_items.reset_index(drop=True)); display(cb_items.reset_index(drop=True))

2024 파일설계서: docs/survey_codebooks/youth_life_survey/2024년_청년삶실태조사_파일설계서.xlsx


,item_no,item,code_2024,label
0,358,외출 빈도,1,직장이나 학교로 평일은 매일 외출한다
1,358,외출 빈도,2,직장이나 학교로 일주일에 3~4일 외출한다
2,358,외출 빈도,3,여가생활을 위해 자주 외출한다
3,358,외출 빈도,4,사람을 만나기 위해 가끔 외출한다
4,358,외출 빈도,5,"보통은 집에 있지만, 자신의 취미생활만을 위해 외출한다"
5,358,외출 빈도,6,"보통은 집에 있지만, 인근 편의점 등에 외출한다"
6,358,외출 빈도,7,자기 방에서 나오지만 집 밖으로는 나가지 않는다
7,358,외출 빈도,8,자기 방에서 거의 나오지 않는다
8,360,은둔 생활 이유,1,학업의 중단으로
9,360,은둔 생활 이유,2,대학 진학의 실패로


,var_2023,code_2023,label
0,sq1a1,1,19-24세
1,sq1a1,2,25-29세
2,sq1a1,3,30-34세
3,sq1a1,4,35-39세
4,sq3,1,자기 방에서 거의 나오지 않는다
5,sq3,2,자기 방에서 나오지만 집 밖으로는 나가지 않는다
6,sq3,3,"보통은 집에 있지만, 인근 편의점 등에 외출한다"
7,sq3,4,"보통은 집에 있지만, 자신의 취미생활만을 위해 외출한다"
8,sq3,5,사람을 만나기 위해 가끔 외출한다
9,sq3,6,여가생활을 위해 자주 외출한다


In [14]:
demand, counts = sv.build_demand()
flow = pd.DataFrame([
    {"step": "1차조사 응답자", "n": counts["screen_rows"]},
    {"step": "정의 충족 (sq3∈{1,2,3,4}, sq3a1∉{5,6}, sq1a1∈{1,2,3})", "n": counts["definition_met_in_screen"]},
    {"step": "은둔 본조사 시트 응답자", "n": counts["hiki_sheet_rows"]},
    {"step": "정의 충족 ∩ 은둔 본조사 (pid 내부 결합) = 수요 테이블", "n": counts["linked_definition_and_hiki_sheet"]},
    {"step": "은둔 본조사 중 정의 미충족", "n": counts["hiki_sheet_not_meeting_definition"]},
])
display(flow)
aux, iso_counts = sv.build_isolated_aux()
iso_flow = pd.DataFrame([
    {"step": "고립 본조사 시트 응답자", "n": iso_counts["iso_sheet_rows"]},
    {"step": "1차조사와 pid 연결", "n": iso_counts["iso_linked_to_screen"]},
    {"step": "19~34세 (sq1a1∈{1,2,3}) = 고립 보조집단", "n": iso_counts["iso_age_19_34"]},
    {"step": "35~39세 제외", "n": iso_counts["iso_age_35_39_excluded"]},
    {"step": "은둔 본조사 시트와 겹치는 pid", "n": iso_counts["overlap_with_hiki_sheet"]},
])
display(iso_flow)
q21_missing = pd.DataFrame({"hikikomori_missing": demand.filter(like="q21_").isna().sum(), "isolated_aux_missing": aux.filter(like="q21_").isna().sum()})
display(q21_missing.T)
display(Markdown(f"**관찰:** 수요 테이블 {len(demand):,}명으로 기대값 4,448명과 일치. 1차조사에서 정의만 적용하면 "
                 f"{counts['definition_met_in_screen']:,}명이고, 은둔 본조사 시트와 pid로 결합해야 4,448명이 된다. "
                 f"고립 보조집단은 은둔 판별 기준(외출·은둔 이유)을 적용하지 않고 연령(19~34세)만 적용해 {len(aux):,}명. Q21의 '.'은 결측으로 바꿨다."))

,step,n
0,1차조사 응답자,21360
1,"정의 충족 (sq3∈{1,2,3,4}, sq3a1∉{5,6}, sq1a1∈{1,2,3})",6402
2,은둔 본조사 시트 응답자,5161
3,정의 충족 ∩ 은둔 본조사 (pid 내부 결합) = 수요 테이블,4448
4,은둔 본조사 중 정의 미충족,713


,step,n
0,고립 본조사 시트 응답자,3713
1,1차조사와 pid 연결,3713
2,"19~34세 (sq1a1∈{1,2,3}) = 고립 보조집단",3032
3,35~39세 제외,681
4,은둔 본조사 시트와 겹치는 pid,0


,q21_01,q21_02,q21_03,q21_04,q21_05,q21_06,q21_07,q21_08,q21_09,q21_10
hikikomori_missing,2,2,4,2,3,2,2,2,1,2
isolated_aux_missing,0,1,0,0,0,0,0,0,0,1


**관찰:** 수요 테이블 4,448명으로 기대값 4,448명과 일치. 1차조사에서 정의만 적용하면 6,402명이고, 은둔 본조사 시트와 pid로 결합해야 4,448명이 된다. 고립 보조집단은 은둔 판별 기준(외출·은둔 이유)을 적용하지 않고 연령(19~34세)만 적용해 3,032명. Q21의 '.'은 결측으로 바꿨다.

**해석과 한계:** 4,448명은 '1차조사 정의 충족'과 '은둔 본조사 응답'을 모두 만족하는 응답자다. 은둔 본조사 응답자 중 713명은 2024 정의를 충족하지 않아 제외된다. 2023 조사는 가중치가 없으므로 이후 비율은 **비가중 응답자 기준**이며 모집단 비율이 아니다. 2024 청년삶의 41%·55% 수치는 원자료가 저장소에 없어 재현하지 않았다.

# PART E. 보조금24 로딩·품질 점검

**목적:** Q2 비교 출처. 이번 단계는 로딩(`dtype=str`, `utf-8-sig`)과 품질 점검만 하고 마스터에 합치지 않는다.

In [15]:
g = {n: pd.read_csv(ROOT / f"data/raw/gov24/gov24_{n}.csv", dtype=str, encoding="utf-8-sig") for n in ["serviceList", "serviceDetail", "supportConditions"]}
ids = {n: set(d["서비스ID"]) for n, d in g.items()}
gov24_quality = pd.DataFrame([{"file": n, "rows": len(d), "cols": d.shape[1], "id_unique": int(d["서비스ID"].nunique()),
                               "id_dup": int(d["서비스ID"].duplicated().sum()), "ids_not_in_list": len(ids[n] - ids["serviceList"]),
                               "cells_missing_pct": round(100 * d.isna().to_numpy().mean(), 2)} for n, d in g.items()])
display(gov24_quality)
key_cols = ["서비스명", "서비스목적요약", "지원대상", "선정기준", "지원내용", "신청방법", "사용자구분", "소관기관명", "온라인신청사이트URL"]
miss = []
for n, d in g.items():
    for c in d.columns:
        if c in key_cols or n == "serviceList":
            miss.append({"file": n, "column": c, "missing": int(d[c].isna().sum()), "missing_pct": round(100 * d[c].isna().mean(), 2)})
gov24_missing = pd.DataFrame(miss).sort_values(["file", "missing_pct"], ascending=[True, False])
display(gov24_missing.head(25))

,file,rows,cols,id_unique,id_dup,ids_not_in_list,cells_missing_pct
0,serviceList,10942,22,10942,0,0,8.32
1,serviceDetail,10942,21,10942,0,0,24.59
2,supportConditions,10942,50,10942,0,0,56.76


,file,column,missing,missing_pct
23,serviceDetail,선정기준,9896,90.44
26,serviceDetail,온라인신청사이트URL,8943,81.73
22,serviceDetail,서비스명,0,0.00
24,serviceDetail,소관기관명,0,0.00
25,serviceDetail,신청방법,0,0.00
27,serviceDetail,지원내용,0,0.00
28,serviceDetail,지원대상,0,0.00
17,serviceList,접수기관,10134,92.62
8,serviceList,선정기준,9896,90.44
15,serviceList,신청방법,3,0.03


**해석과 한계:** 행 수·ID 중복·목록 대비 ID 포함 여부와 결측만 확인했다. SSIS와의 서비스 매칭, 내용 비교는 Q2 단계에서 한다. 해시 불일치(PART A)는 미해결 사항으로 남긴다.

# 저장

In [16]:
OUT = ROOT / "outputs/00_data_preparation"
(OUT / "tables").mkdir(parents=True, exist_ok=True)
P = ROOT / "data/processed"; P.mkdir(parents=True, exist_ok=True)
sm.save_master(master)   # services_master.csv + services_master.meta.json (규격: docs/services_master_spec.md)
demand.to_csv(P / "demand_2023_hikikomori.csv", index=False, encoding="utf-8-sig")
aux.to_csv(P / "demand_2023_isolated_aux.csv", index=False, encoding="utf-8-sig")
tables = {"raw_hash_check": hash_check, "ssis_merge_check": merge_check, "ssis_tag_source_check": tag_source_check,
          "ssis_text_clean_check": clean_check, "label_value_counts": label_value_counts, "need_area_rules": rules,
          "app_domain_map_draft": dom_map, "app_domain_map_v2": map_v2, "label_revision_diff": label_revision_diff, "hikikomori_definition_flow": flow, "isolated_aux_flow": iso_flow, "survey_q21_missing": q21_missing.reset_index(names="item"),
          "gov24_quality": gov24_quality, "gov24_missing": gov24_missing}
for k, v in tables.items():
    v.to_csv(OUT / "tables" / f"{k}.csv", index=False, encoding="utf-8-sig")
chk = pd.read_csv(P / "services_master.csv", dtype=str, encoding="utf-8-sig", keep_default_na=False)
print("저장 확인:", chk.shape, "| demand", pd.read_csv(P / "demand_2023_hikikomori.csv", dtype=str).shape, "| tables", sorted(p.name for p in (OUT / "tables").glob("*.csv")))

저장 확인: (5288, 153) | demand (4448, 15) | tables ['app_domain_map_draft.csv', 'app_domain_map_v2.csv', 'apply_outing_basis_counts.csv', 'gov24_missing.csv', 'gov24_quality.csv', 'hikikomori_definition_flow.csv', 'isolated_aux_flow.csv', 'label_revision_diff.csv', 'label_revision_v3_diff.csv', 'label_value_counts.csv', 'need_area_rules.csv', 'outing_v3_diff.csv', 'raw_hash_check.csv', 'ssis_merge_check.csv', 'ssis_tag_source_check.csv', 'ssis_text_clean_check.csv', 'survey_q21_missing.csv', 'youth_scope_counts.csv']


## 테스트
`tests/`(`test_step00_preprocess.py`, `test_v3_youth_scope.py`)를 실행한다(행 수, 키 중복, 4,448명 재현, 원본 해시 불변, 라벨 값 범위, 근거 없는 값 금지, outing_burden 공식, 다집단 예시, 영역 필수 제외).

In [17]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], capture_output=True, text=True, cwd=ROOT,
                   env={**os.environ, "PYTHONIOENCODING": "utf-8"})
test_tail = (r.stdout.strip().splitlines() or [""])[-1]
print(test_tail); assert r.returncode == 0, r.stdout[-3000:]

128 passed in 30.54s


## HTML 요약
사용자 요청(2026-11-02)으로 HTML 작업을 보류한다. `BUILD_HTML = False`이면 기존 HTML을 다시 쓰지 않으므로, `outputs/00_data_preparation/summary/`의 HTML은 보완 전 실행 결과다.

In [18]:
BUILD_HTML = False
from src.preprocess import summary_html as H
lv = label_value_counts.pivot_table(index=["attribute", "value"], columns="source", values="n", fill_value=0, aggfunc="sum").reset_index()
if BUILD_HTML:
  H.build(OUT / "summary/00_data_preparation_summary_final.html",
    title="STEP 00 데이터 준비",
    lead=f"SSIS {len(master):,}개 서비스의 공통 마스터와 은둔 청년 {len(demand):,}명 수요 테이블을 재현했다.",
    summary="목록·상세를 servId로 1:1 병합하고, 자격·접근성·추천 요소·영역의 규칙 후보 값을 근거 문장과 함께 채웠다. "
            "규칙 값은 후보이며 정밀도는 아직 측정하지 않았다. 2023 조사는 가중치가 없어 비율은 응답자 기준이다.",
    meta=["입력: data/raw/ssis, gov24, isolated_youth_survey_2023", f"서비스 {len(master):,}행 · 키 중복 0", f"은둔 {len(demand):,}명",
          "수집일 2026-10-30 / 실행 2026-11-02", f"테스트: {test_tail}"],
    sections=[
      {"id": "s1", "title": "목적과 범위", "blocks": [("p", "분석(Q1~Q3)과 LLM 추천이 같은 서비스 테이블을 쓰도록 서비스 1행 마스터를 만든다. "
          "LLM 라벨·사람 검수·Q1 본 분석·정답셋 판정은 이 단계에 포함하지 않는다.")]},
      {"id": "s2", "title": "핵심 결과", "blocks": [
          ("box", "result", f"병합 {len(master):,}행(중앙 461 + 지자체 4,827), servId 중복 0. 은둔 정의 적용 후 은둔 본조사와 결합한 응답자 {len(demand):,}명으로 기대값과 일치."),
          ("table", merge_check, "표 1. SSIS 목록·상세 병합 점검", "outputs/00_data_preparation/tables/ssis_merge_check.csv"),
          ("table", flow, "표 2. 은둔 정의 적용 흐름 (비가중 응답자 수)", "outputs/00_data_preparation/tables/hikikomori_definition_flow.csv")]},
      {"id": "s3", "title": "데이터·방법", "blocks": [
          ("p", "텍스트는 \\r을 줄바꿈으로 바꾸고 HTML 엔티티를 해제했다. 중첩 리스트는 평탄화·JSON 형태를 합쳤다. 라벨은 구조화 필드 → 키워드 규칙 순서로 채우고, 각 값에 근거 문장·방법·검수 여부를 붙였다. 사전은 src/preprocess/keywords.py에 있다."),
          ("table", clean_check, "표 3. 정제 점검", "outputs/00_data_preparation/tables/ssis_text_clean_check.csv"),
          ("table", dom_map, "표 4. need_area_q21 → app_domain 대응표 (초안, 미확정)", "outputs/00_data_preparation/tables/app_domain_map_draft.csv")]},
      {"id": "s4", "title": "규칙 라벨 후보 분포", "blocks": [
          ("table", lv, "표 5. 라벨 값별 서비스 수 (출처별, 규칙 후보)", "outputs/00_data_preparation/tables/label_value_counts.csv"),
          ("box", "warning", "이 분포는 규칙 후보의 분포다. 원문 키워드로 정한 값은 모두 검수 대상(review=True)이며, 서비스 수를 공급 규모나 수급률로 해석하지 않는다.")]},
      {"id": "s5", "title": "보조금24 품질 점검", "blocks": [
          ("table", gov24_quality, "표 6. 보조금24 파일별 품질", "outputs/00_data_preparation/tables/gov24_quality.csv"),
          ("table", hash_check, "표 7. 원자료 해시 (보조금24는 수집 로그와 대조)", "outputs/00_data_preparation/tables/raw_hash_check.csv"),
          ("box", "warning", "보조금24 목록·상세 파일은 수집 로그의 해시·바이트 수와 다르다. 원인을 확인하기 전까지 Q2 비교 입력으로 확정하지 않는다.")]},
      {"id": "s6", "title": "한계와 다음 단계", "blocks": [
          ("p", "연령 추출은 AND/OR 조건을 구분하지 못하고 출생연도 표기를 다루지 않는다. 신청 장벽은 가장 쉬운 경로를 택하므로 서류 제출용 방문을 놓칠 수 있다. job_relevance의 0(무관)은 근거 문장이 없어 채우지 않았다."),
          ("p", "다음: STEP 01에서 태그 품질, 청년 후보 범위, 영역·접근성·추천 요소 분포와 N 비율을 보고 라벨 방법을 정한다. STEP 02에서 규칙 정밀도를 사람 검수로 측정한다.")]},
    ],
    footer="근거 노트북: notebooks/00_data_preparation.ipynb · 표: outputs/00_data_preparation/tables/ · 코드: src/preprocess/ · 검증: tests/test_step00_preprocess.py, logs/validation/raw_hashes_step00.json")
print("HTML 생성" if BUILD_HTML else "HTML 보류 (BUILD_HTML=False)")

HTML 보류 (BUILD_HTML=False)


# 최종 완료 확인

아래 셀은 실제 생성 파일을 다시 확인한다. 인수조건 판정은 이 노트북을 초기 커널에서 위에서 아래로 실행한 결과 기준이다.

In [19]:
files = [p.relative_to(ROOT).as_posix() for p in sorted((ROOT / "outputs/00_data_preparation").rglob("*")) if p.is_file()]
files += [f"data/processed/{n}" for n in ["services_master.csv", "demand_2023_hikikomori.csv", "demand_2023_isolated_aux.csv"] if (ROOT / "data/processed" / n).exists()]
files += ["logs/validation/raw_hashes_step00.json"] if raw_manifest.MANIFEST.exists() else []
display(pd.DataFrame({"file": files}))

,file
0,outputs/00_data_preparation/review/boundary_cases_v2.csv
1,outputs/00_data_preparation/review/boundary_cases_v2_review.csv
2,outputs/00_data_preparation/review/label_review_fix_diff.csv
3,outputs/00_data_preparation/review/outing_need_N_causes.csv
4,outputs/00_data_preparation/tables/app_domain_map_draft.csv
5,outputs/00_data_preparation/tables/app_domain_map_v2.csv
6,outputs/00_data_preparation/tables/apply_outing_basis_counts.csv
7,outputs/00_data_preparation/tables/gov24_missing.csv
8,outputs/00_data_preparation/tables/gov24_quality.csv
9,outputs/00_data_preparation/tables/hikikomori_definition_flow.csv


| # | 인수조건 | 판정 | 근거 |
|---|---|---|---|
| 1 | 범위·입출력 확정 | 통과 | 시작 셀 출력 목록 = 위 파일 목록. 그림은 해당 없음(사유: 전처리 단계) |
| 2 | 노트북 형식·환경 | 통과 | nbformat v4, python3 커널, 버전은 첫 코드 셀 출력 |
| 3 | 실행·출력 보존 | 통과 | `jupyter nbconvert --execute`로 초기 커널 순차 실행, 오류 시 실행 중단 |
| 4 | 정의·결정 일관성 | 통과 | Decision Box 1~5, 코드 위치 명시 |
| 5 | 결과표 검증 | 통과 | 저장 셀에서 다시 읽어 행 수 확인, pytest 통과 |
| 6 | 결과그림 검증 | 해당 없음 | 이 단계는 그림을 만들지 않음 |
| 7 | 매체 간 정합성 | 통과 | HTML 표는 같은 DataFrame에서 생성 |
| 8 | HTML 독립성·탐색 | 보류 | 사용자 요청으로 HTML 작업 보류. 기존 HTML은 보완 전 결과 |
| 9 | 화면·접근성·인쇄 | 보류 | 위와 같음 |
| 10 | 해석·공유 적합성 | 통과 | 규칙 값=후보, 비가중, 서비스 수≠공급 규모 명시. 개인 응답 원자료는 HTML에 없음 |
| 11 | 최종 인수 기록 | 통과 | 미해결: 보조금24 해시 불일치, 규칙 정밀도 미측정, 연령 AND/OR, 대응표 팀 확인 |

8·9번(HTML)이 보류이므로 이 단계는 **부분 완료**다.